In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score

In [2]:
path_to_repo = Path('..').resolve()
path_to_data = path_to_repo / 'data'

In [3]:
application = pd.read_csv(path_to_data / 'application_record.csv')
credit = pd.read_csv(path_to_data / 'credit_record.csv')

In [4]:
application = application.drop_duplicates("ID")

In [5]:
application["CODE_GENDER"] = application["CODE_GENDER"].map({"F":0, "M":1})
application["FLAG_OWN_CAR"] = application["FLAG_OWN_CAR"].map({"N":0, "Y":1})
application["FLAG_OWN_REALTY"] = application["FLAG_OWN_REALTY"].map({"N":0, "Y":1})

In [6]:
if "OCCUPATION_TYPE" in application.columns:
    application = application.drop(columns=["OCCUPATION_TYPE"])

In [7]:
application["AGE"] = (-application["DAYS_BIRTH"] / 365).astype(int)

application["EXPERIENCE"] = application["DAYS_EMPLOYED"].apply(
    lambda x: 0 if x > 0 else int(-x/365)
)

In [8]:
application["CNT_FAM_MEMBERS"] = application["CNT_FAM_MEMBERS"].round().astype(int)

In [9]:
cat_cols = [
    "NAME_INCOME_TYPE",
    "NAME_EDUCATION_TYPE",
    "NAME_FAMILY_STATUS",
    "NAME_HOUSING_TYPE"
]

for col in cat_cols:
    application[col] = application[col].astype("category").cat.codes

In [10]:
credit["bad"] = credit["STATUS"].apply(lambda x: 1 if x in ["2","3","4","5"] else 0)

client_status = credit.groupby("ID")["bad"].max().reset_index()

In [11]:
data = application.merge(client_status, on="ID", how="inner")

In [12]:
data["LOG_INCOME"] = np.log1p(data["AMT_INCOME_TOTAL"])
data = data.drop(columns=["AMT_INCOME_TOTAL"])

In [13]:
data["LOG_INCOME_PER_PERSON"] = data["LOG_INCOME"] / data["CNT_FAM_MEMBERS"]

In [14]:
data["DEPENDENCY_RATIO"] = data["CNT_CHILDREN"] / (
    data["CNT_FAM_MEMBERS"] - data["CNT_CHILDREN"] + 1e-6
)

In [15]:
data = data.drop(columns=["DAYS_BIRTH", "DAYS_EMPLOYED"])

In [16]:
def exp_bin(x):
    if x <= 0: return 0
    if x <= 5: return 1
    if x <= 10: return 2
    if x <= 20: return 3
    return 4

data["EXPERIENCE_BIN"] = data["EXPERIENCE"].apply(exp_bin)

In [17]:
def age_bin(a):
    if a < 25: return 0
    if a < 35: return 1
    if a < 50: return 2
    if a < 65: return 3
    return 4

data["AGE_BIN"] = data["AGE"].apply(age_bin)

In [18]:
data["AGE_X_INCOME"] = data["AGE"] * data["LOG_INCOME"]
data["EXP_X_INCOME"] = data["EXPERIENCE"] * data["LOG_INCOME"]

In [19]:
def winsorize(s):
    return s.clip(s.quantile(0.01), s.quantile(0.99))

for col in ["LOG_INCOME", "LOG_INCOME_PER_PERSON", "EXPERIENCE"]:
    data[col] = winsorize(data[col])

In [20]:
def winsorize(s):
    return s.clip(s.quantile(0.01), s.quantile(0.99))

for col in ["LOG_INCOME", "LOG_INCOME_PER_PERSON", "EXPERIENCE"]:
    data[col] = winsorize(data[col])

In [21]:
data["HIGH_INCOME_FLAG"] = (data["LOG_INCOME"] > data["LOG_INCOME"].quantile(0.98)).astype(int)

In [22]:
counts = application["NAME_INCOME_TYPE"].value_counts(normalize=True)
rare = counts[counts < 0.005].index
application["NAME_INCOME_TYPE"] = application["NAME_INCOME_TYPE"].replace(
    rare, "Other"
)

In [23]:
data["INCOME_BIN"] = pd.qcut(data["LOG_INCOME"], q=4, labels=False)

In [24]:
data["ADULTS"] = data["CNT_FAM_MEMBERS"] - data["CNT_CHILDREN"]
data["ADULTS"] = data["ADULTS"].clip(lower=1)

In [25]:
data["EXP_AGE_RATIO"] = data["EXPERIENCE"] / (data["AGE"] + 1e-6)

In [26]:
data["STABILITY_SCORE"] = (
    -0.3 * data["DEPENDENCY_RATIO"] +
     0.4 * data["EXPERIENCE_BIN"] +
     0.3 * (4 - data["AGE_BIN"])
)

In [27]:
data["CHILD_BIN"] = pd.cut(
    data["CNT_CHILDREN"],
    bins=[0,1,3,10],
    labels=[0,1,2]
)

In [28]:
data['CHILD_BIN'].value_counts()

CHILD_BIN
0    7492
1    3675
2      85
Name: count, dtype: int64

In [29]:
data.shape

(36457, 30)

In [30]:
data["bad"].value_counts()

bad
0    35841
1      616
Name: count, dtype: int64

In [31]:
data

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,FLAG_MOBIL,...,EXPERIENCE_BIN,AGE_BIN,AGE_X_INCOME,EXP_X_INCOME,HIGH_INCOME_FLAG,INCOME_BIN,ADULTS,EXP_AGE_RATIO,STABILITY_SCORE,CHILD_BIN
0,5008804,1,1,1,0,4,1,0,4,1,...,3,1,414.902781,155.588543,0,3,2,0.375000,2.1,NaN
1,5008805,1,1,1,0,4,1,0,4,1,...,3,1,414.902781,155.588543,0,3,2,0.375000,2.1,NaN
2,5008806,1,1,1,0,4,4,1,1,1,...,1,3,674.581609,34.892152,0,0,2,0.051724,0.7,NaN
3,5008808,0,0,1,0,0,4,3,1,1,...,2,3,650.321409,100.049448,0,3,1,0.153846,1.1,NaN
4,5008809,0,0,1,0,0,4,3,1,1,...,2,3,650.321409,100.049448,0,3,1,0.153846,1.1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36452,5149828,1,1,1,0,4,4,1,1,1,...,2,2,595.035561,75.961987,0,3,2,0.127660,1.4,NaN
36453,5149834,0,0,1,0,0,1,1,1,1,...,1,1,394.917174,35.901561,0,1,2,0.090909,1.3,NaN
36454,5149838,0,0,1,0,1,1,1,1,1,...,1,1,394.917174,35.901561,0,1,2,0.090909,1.3,NaN
36455,5150049,0,0,1,0,4,4,1,1,1,...,1,2,615.193576,12.554971,0,3,2,0.020408,1.0,NaN


In [32]:
data.to_csv(path_to_data / "clean_data.csv", index=False)